# Target Selection task API examples

This notebook demonstrates the independent task-level API. Each section can be run separately: target collection, source merging, local visibility, Rubin coverage/photometry, and reports. The cells that access MOP, TAP, or Butler are intentionally not executed automatically because they require an RSP session and can be expensive.

## 1. Imports and run settings

Change these variables directly in the notebook. No monolithic `run_analysis` call is required.

In [1]:
from pathlib import Path

import pandas as pd

from target_selection import (
    collect_mop_targets,
    load_target_list,
    merge_targets,
    restrict_targets,
    evaluate_visibility,
    save_visibility_plots,
    make_visibility_sequence,
    query_lsst_coverage,
    compute_lsst_photometry,
    save_lsst_photometry,
    create_target_report,
    create_target_reports,
    create_lightcurves_report,
)
from target_selection.run_paths import create_run_structure
from target_selection.sources.lsst import create_tap_service, create_butler

START_DATE = "2026-08-17"
END_DATE = START_DATE
OBSERVATORY = "El Leoncito"
OUTPUT_DIR = Path("outputs/task_api_example")
DATA_RELEASE = "DP2"
MAX_WORKERS = 4

paths = create_run_structure(OUTPUT_DIR, START_DATE, END_DATE, data_release=DATA_RELEASE)
paths

{'run': PosixPath('outputs/task_api_example/2026-08-17__obs_18-00_to_06-00'),
 'tables': PosixPath('outputs/task_api_example/2026-08-17__obs_18-00_to_06-00/tables'),
 'sky_plots': PosixPath('outputs/task_api_example/2026-08-17__obs_18-00_to_06-00/sky_plots'),
 'visibility_plots': PosixPath('outputs/task_api_example/2026-08-17__obs_18-00_to_06-00/visibility_plots'),
 'monitoring_reports': PosixPath('outputs/task_api_example/2026-08-17__obs_18-00_to_06-00/monitoring_reports'),
 'target_reports': PosixPath('outputs/task_api_example/target_reports'),
 'targets': PosixPath('outputs/task_api_example/target_reports'),
 'targets_visibility_selected': PosixPath('outputs/task_api_example/target_reports')}

## 2. Collect targets from providers

MOP is optional. A local CSV can be loaded with a column map when its names are not already normalized. Follow-up inventories such as HSH can be imported through the configured source adapters or the registry workflow.

In [2]:
# Requires an initialized MOP client or an active RSP/MOP environment.
mop_targets = collect_mop_targets(
    start_date=START_DATE, end_date=END_DATE, observatory=OBSERVATORY,
    cache_dir=OUTPUT_DIR, max_workers=MAX_WORKERS,
)

# Example local provider list. Required normalized fields: Target, RA_deg, Dec_deg.
csv_targets = load_target_list(
    "/home/karennowo/target_selection/hsh_data/hsh_objects.csv",
    source_name="hsh",
    column_map={"Target": "object", "RA_deg": "ra_deg", "Dec_deg": "dec_deg"}
)

# Combine any number of provider/survey target tables. Higher coordinate priority wins.
targets = merge_targets(mop_targets, csv_targets)
targets = restrict_targets(targets, target_names=["OGLE-2025-BLG-0451"])


## 3. Evaluate local visibility

The visibility task applies astronomical-night and observing-window constraints, then the altitude and minimum-duration selection criteria.

In [3]:
visibility = evaluate_visibility(
    targets, start_date=START_DATE, end_date=END_DATE,
    observatory=OBSERVATORY, minimum_altitude_deg=40.0,
    minimum_observable_minutes=90.0, time_step_minutes=1,
    observing_windows=("20:00", "07:00"),
    output_path=paths["visibility_plots"] / "visibility_selection.csv",
)
visibility.head()

# Save nightly plots, optionally limiting each panel to a maximum number of targets.
nightly_plots = save_visibility_plots(
    targets, start_date=START_DATE, end_date=END_DATE,
    output_dir=paths["visibility_plots"], observatory=OBSERVATORY,
    minimum_altitude_deg=40.0, minimum_observable_minutes=90.0,
    observing_windows=("20:00", "07:00"), max_targets_per_plot=20,
)

# Stack the already computed visibility table into a PDF or PNG sequence.
sequence = make_visibility_sequence(
    visibility, paths["visibility_plots"] / "visibility_sequence.pdf",
    observatory=OBSERVATORY, observing_windows=("20:00", "07:00"),
    start_date=START_DATE, end_date=END_DATE,
)


      visibility: 1/1 (2026-08-17, 0/0 selected, 1 plot(s))


ValueError: No valid observing dates were found.

## 4. Add Rubin/LSST context

TAP returns visit/detector coverage. Butler is used for image/coadd products and the configured photometry method. These are optional tasks, so the same target table can be analyzed without a reference collection.

In [ ]:
tap_service = create_tap_service(DATA_RELEASE)
coverage = query_lsst_coverage(
    targets, tap_service=tap_service, data_release=DATA_RELEASE,
    max_workers=MAX_WORKERS,
)
coverage.to_csv(paths["tables"] / "release_coverage.csv", index=False)

# Select one method in the call: dia_forced_catalog, coadd_forced, or calexp_forced.
release_photometry = compute_lsst_photometry(
    targets, coverage=coverage, method="dia_forced_catalog",
    tap_service=tap_service, data_release=DATA_RELEASE,
    max_workers=MAX_WORKERS,
)
save_lsst_photometry(release_photometry, paths["tables"] / "release_photometry.csv")


## 5. Generate products

Reports can be generated for all targets or for one selected target. Individual target reports are stored in the shared `target_reports` folder.

In [ ]:
target_reports = create_target_reports(
    targets, coverage=coverage, data_release=DATA_RELEASE,
    release_photometry=release_photometry, output_dir=paths["target_reports"],
    max_workers=MAX_WORKERS, overwrite=False,
)

# One target only:
one_target = targets.loc[targets["Target"].eq("OGLE-2025-BLG-0451")].iloc[0]
report = create_target_report(
    one_target, coverage=coverage, data_release=DATA_RELEASE,
    release_photometry=release_photometry, output_dir=paths["target_reports"],
)

# A sky map is a product task too; metrics are columns in the merged target table.
plot_sky_dual_metric(
    targets, "mag_now", "release_n_visits",
    paths["sky_plots"] / "sky_by_mag_and_visits.png",
    title=f"{DATA_RELEASE} target coverage", marker_encoding="split_color",
)

# Light curves can be generated from the normalized provider/reference tables.\n",

lightcurves = create_lightcurves_report(
    targets, output_path=paths["monitoring_reports"] / "lightcurves.pdf",
    data_release=DATA_RELEASE, release_photometry=release_photometry,
)


## 6. Inspect the run

The run directory contains only the products requested by the cells. The shared target database/cache can be reused by later runs.

In [ ]:
# for path in sorted(paths["run"].rglob("*")):
#     if path.is_file():
#         print(path)
